# HPO SolarSystemLander

Study Series 2A (8D) and 2B (11D) from `hpo/design3.md`.

## Setup

In [ ]:
# Colab setup
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r hpo/requirements.txt

from pathlib import Path
import sys
from google.colab import drive

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")
drive.mount("/content/drive")
HPO_STUDY_DIR = Path("/content/drive/MyDrive/rl_lab/hpo")
HPO_STUDY_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
from dataclasses import replace
import torch

from dqn.vector_training import VectorTrainingConfig
from hpo.evaluation.reporting import show_lander_live_progress
from hpo.evaluation.scoring import ScoringConfig
from hpo.objective import TrialConfig
from hpo.solar_system_lander.environment import SolarSystemLanderEnvironmentFactory
from hpo.study import neighbors, run_study, select_robust_best

OBSERVATION_MODE = "8d"  # Series 2A; use "11d" for Series 2B
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRIAL_CFG = TrialConfig(num_envs=20, device=device)
SCORING_CFG = ScoringConfig(quality_weight=0.7, eval_episodes=10)
ENVIRONMENT_FACTORY = SolarSystemLanderEnvironmentFactory(OBSERVATION_MODE)
STORAGE_PATH = HPO_STUDY_DIR / f"solar_system_lander_{OBSERVATION_MODE}.db"
device, STORAGE_PATH

## Search spaces

In [ ]:
BATCH_SIZES = [256, 512, 1024]
LEARNING_STARTS = [1_000, 2_500, 5_000]
OPTIMIZE_EVERY = [2, 4, 8]
REPLAY_CAPACITIES = [100_000, 200_000, 400_000]
EPISODE_COUNTS = [500, 1_000, 1_500]

BASELINE_PARAMS = {
    "learning_rate": 0.001606,
    "batch_size": 1024,
    "eps_end": 0.0197,
    "eps_decay": 8_446,
    "gamma": 0.99,
    "tau": 0.005,
    "learning_starts": 2_500,
    "optimize_every": 4,
    "replay_memory_capacity": 200_000,
    "num_episodes": 600,
}

def vector_config(params):
    return VectorTrainingConfig(
        num_episodes=params["num_episodes"],
        batch_size=params["batch_size"],
        gamma=0.99,
        eps_start=1.0,
        eps_end=params["eps_end"],
        eps_decay=params["eps_decay"],
        tau=0.005,
        learning_rate=params["learning_rate"],
        learning_starts=params["learning_starts"],
        optimize_every=params["optimize_every"],
    )

class SearchSpace0:
    def training_config(self, trial):
        return vector_config(BASELINE_PARAMS)

    def replay_memory_capacity(self, trial):
        return BASELINE_PARAMS["replay_memory_capacity"]

class SearchSpace1:
    def training_config(self, trial):
        return vector_config(BASELINE_PARAMS | {
            "learning_rate": trial.suggest_float("learning_rate", 5e-4, 3e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", BATCH_SIZES),
            "learning_starts": trial.suggest_categorical("learning_starts", LEARNING_STARTS),
            "optimize_every": trial.suggest_categorical("optimize_every", OPTIMIZE_EVERY),
            "num_episodes": trial.suggest_categorical("num_episodes", EPISODE_COUNTS),
        })

    def replay_memory_capacity(self, trial):
        return BASELINE_PARAMS["replay_memory_capacity"]

class SearchSpace2:
    def __init__(self, best_s1):
        self.params = BASELINE_PARAMS | best_s1

    def training_config(self, trial):
        return vector_config(self.params | {
            "eps_end": trial.suggest_float("eps_end", 0.01, 0.10),
            "eps_decay": trial.suggest_int("eps_decay", 5_000, 100_000, log=True),
        })

    def replay_memory_capacity(self, trial):
        return BASELINE_PARAMS["replay_memory_capacity"]

class SearchSpace3:
    def __init__(self, best_s1, best_s2):
        self.params = BASELINE_PARAMS | best_s1 | best_s2

    def training_config(self, trial):
        return vector_config(self.params)

    def replay_memory_capacity(self, trial):
        return trial.suggest_categorical("replay_memory_capacity", REPLAY_CAPACITIES)

class SearchSpace4:
    def __init__(self, best_s1, best_s2, best_s3):
        self.params = BASELINE_PARAMS | best_s1 | best_s2 | best_s3

    def training_config(self, trial):
        p = self.params
        return vector_config(p | {
            "learning_rate": trial.suggest_float("learning_rate", p["learning_rate"] / 2, p["learning_rate"] * 2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", neighbors(p["batch_size"], BATCH_SIZES)),
            "eps_end": trial.suggest_float("eps_end", max(0.01, p["eps_end"] - 0.02), min(0.10, p["eps_end"] + 0.02)),
            "eps_decay": trial.suggest_int("eps_decay", max(1, p["eps_decay"] // 2), p["eps_decay"] * 2, log=True),
            "learning_starts": trial.suggest_categorical("learning_starts", neighbors(p["learning_starts"], LEARNING_STARTS)),
            "optimize_every": trial.suggest_categorical("optimize_every", neighbors(p["optimize_every"], OPTIMIZE_EVERY)),
            "num_episodes": trial.suggest_categorical("num_episodes", neighbors(p["num_episodes"], EPISODE_COUNTS)),
        })

    def replay_memory_capacity(self, trial):
        return self.params["replay_memory_capacity"]

## Optimization

In [ ]:
def run_lander_study(
    study_name, search_space, n_trials, scoring_cfg,
    previous_studies=(),
):
    def show_progress(study, *, target_trials):
        show_lander_live_progress(
            study,
            target_trials=target_trials,
            lander_studies=[*previous_studies, study],
        )

    study = run_study(
        study_name=study_name,
        search_space=search_space,
        n_trials=n_trials,
        storage_path=STORAGE_PATH,
        environment_factory=ENVIRONMENT_FACTORY,
        trial_cfg=TRIAL_CFG,
        scoring_cfg=scoring_cfg,
        study_attrs=ENVIRONMENT_FACTORY.metadata(),
        progress_fn=show_progress,
    )
    if study.best_trial.params:
        best_params = select_robust_best(
            study=study,
            search_space=search_space,
            environment_factory=ENVIRONMENT_FACTORY,
            trial_cfg=TRIAL_CFG,
            scoring_cfg=scoring_cfg,
            base_seed=TRIAL_CFG.seed,
            extra_seeds=(1001,),
        )
    else:
        best_params = study.user_attrs["robust_best_params"]
    show_progress(study, target_trials=n_trials)
    return study, best_params

study0, _ = run_lander_study("s0_qe_baseline", SearchSpace0(), 3, SCORING_CFG)
study0.set_user_attr("baseline_params", BASELINE_PARAMS)
scoring_cfg = replace(
    SCORING_CFG,
    baseline_env_steps=study0.user_attrs["baseline_env_steps"],
    baseline_processed_samples=study0.user_attrs["baseline_processed_samples"],
)
study1, best_s1 = run_lander_study("s1_qe_update_economy", SearchSpace1(), 40, scoring_cfg, [study0])
study2, best_s2 = run_lander_study("s2_qe_exploration", SearchSpace2(best_s1), 25, scoring_cfg, [study0, study1])
study3, best_s3 = run_lander_study("s3_qe_replay_capacity", SearchSpace3(best_s1, best_s2), 10, scoring_cfg, [study0, study1, study2])
study4, best_s4 = run_lander_study("s4_qe_joint_finetune", SearchSpace4(best_s1, best_s2, best_s3), 25, scoring_cfg, [study0, study1, study2, study3])

## Reload studies after a runtime reset

In [ ]:
# Run only after the runtime environment has been reset.
import optuna

def load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{STORAGE_PATH}")

study0 = load_study("s0_qe_baseline")
study1 = load_study("s1_qe_update_economy")
study2 = load_study("s2_qe_exploration")
study3 = load_study("s3_qe_replay_capacity")
study4 = load_study("s4_qe_joint_finetune")

## Analysis

In [ ]:
import pandas as pd

studies = [study0, study1, study2, study3, study4]
labels = ["S0", "S1", "S2", "S3", "S4"]

def selected_trial(study):
    params = study.user_attrs.get("robust_best_params")
    candidates = [trial for trial in study.trials if trial.params == params]
    return max(candidates, key=lambda trial: trial.value) if candidates else study.best_trial

rows = []
for label, study in zip(labels, studies):
    trial = selected_trial(study)
    rows.append({
        "study": label,
        "objective_score": study.user_attrs.get("robust_best_objective_score", trial.value),
        "gym_score": study.user_attrs.get("robust_best_gym_score", trial.user_attrs["gym_score"]),
        "training_effort": study.user_attrs.get("robust_best_training_effort", trial.user_attrs["training_effort"]),
        **trial.user_attrs.get("gym_scores", {}),
    })

display(pd.DataFrame(rows).set_index("study").style.format("{:.1f}"))